# Exercise 4.2 — Fitting a logistic regression model

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-02-logistic-regression.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

Linear regression on Day 3 predicted a number, glucose. Logistic regression
predicts a **probability**: the chance that a person has diabetes, given their
glucose. A probability must stay between 0 and 1, and a straight line does not.
The trick is to fit the straight line on a different scale, the **log-odds**,
and convert back.

**Theory:** probability, odds and log-odds; the logistic (sigmoid) function;
reading a logistic regression coefficient; from a fitted model to a
probability.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

The same 750 people and the same split as Exercise 4.1: every fifth person is
held back as the test set.

In [ ]:
diabetes = pd.read_csv("data/diabetes.csv")
diabetes = diabetes[["glucose", "mass", "age", "diabetes"]]
diabetes = diabetes[(diabetes["glucose"] > 0) & (diabetes["mass"] > 0)]   # remove the impossible zeros
diabetes = diabetes.reset_index(drop=True)
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

------------------------------------------------------------------------

### Part 1 — Probability, odds and log-odds

Before fitting any model we need to be clear about how three ways of describing
the same chance are related:

- **Probability** $p$: the chance of the event happening, between 0 and 1.
- **Odds**: the probability of the event divided by the probability of no event,
  $$\text{odds} = \frac{p}{1-p}$$
  between 0 and infinity.
- **Log-odds**, also called the **logit**: the natural log of the odds,
  $$\text{logit}(p) = \log\left(\frac{p}{1-p}\right)$$
  which can be any number, negative or positive.

Start from a probability of 0.8:

In [ ]:
prob     = 0.8
odds     = prob / (1 - prob)
log_odds = np.log(odds)

print("probability:", prob)
print("odds:       ", round(odds, 3))
print("log-odds:   ", round(log_odds, 3))

A probability of 0.8 means odds of 4 ("four to one"), and log-odds of 1.386.

**Complete this table** on paper, using a calculator for the logs. The first row
is done for you.

| Probability $p$ | Odds $p/(1-p)$ | Log-odds $\log(\text{odds})$ |
|:-:|:-:|:-:|
| 0.8 | 4 | 1.386 |
| 0.5 | | |
| 0.2 | | |
| 0.9 | | |

Then check it in code:

In [ ]:
# Replace every ______ below, then run this cell.
table = pd.DataFrame({"prob": [0.8, 0.5, 0.2, 0.9]})
table["odds"]     = ______
table["log_odds"] = np.log(table["odds"])

table.round(3)

Going back the other way, from log-odds $z$ to probability, is the **logistic**
(or **sigmoid**) function:

$$p = \frac{1}{1 + e^{-z}}$$

It takes any number and squeezes it into the range 0 to 1:

In [ ]:
log_odds = np.linspace(-6, 6, 121)
probability = 1 / (1 + np.exp(-log_odds))

fig, ax = plt.subplots()
ax.plot(log_odds, probability, color="red", linewidth=2)
ax.axhline(0, color="grey", linestyle="--")
ax.axhline(1, color="grey", linestyle="--")
ax.set_xlabel("Log-odds z")
ax.set_ylabel("Probability p = 1 / (1 + exp(-z))")
plt.show()

> **Learning points**
>
> - Probability lies between 0 and 1, odds between 0 and infinity, log-odds anywhere.
> - A probability of 0.5 is odds of 1 and log-odds of 0.
> - Logistic regression is a straight line on the log-odds scale; the sigmoid turns it back into a probability.

------------------------------------------------------------------------

### Part 2 — Fit the model

Check the class balance of the training rows first:

In [ ]:
train["diabetes"].value_counts()

220 of the 600 have diabetes (1), 36.7%. The model predicts the probability of
1. `C=np.inf` switches off scikit-learn's default penalty, so the coefficients
match R's.

In [ ]:
logistic_fit_glucose = LogisticRegression(C=np.inf)
logistic_fit_glucose = logistic_fit_glucose.fit(train[["glucose"]], train["diabetes"])

intercept = logistic_fit_glucose.intercept_[0]
slope     = logistic_fit_glucose.coef_[0, 0]

print("intercept:", round(intercept, 4))
print("slope:    ", round(slope, 4))

This corresponds to the model

$$\text{logit}\big(P(\text{diabetes} = \text{pos} \mid \text{glucose})\big)
= \beta_0 + \beta_1 \times \text{glucose}$$

and with the fitted coefficients

$$\text{log-odds of diabetes} = -5.064 + 0.0350 \times \text{glucose}$$

To get the probability for a given glucose, apply the sigmoid:

$$\hat{p} = \frac{1}{1 + \exp\big(-(\beta_0 + \beta_1 \times \text{glucose})\big)}$$

**Reading the slope.** Each extra mg/dL of glucose adds 0.0350 to the log-odds
of diabetes. Log-odds are hard to picture, so turn it into odds: adding 0.0350
to the log-odds multiplies the odds by $e^{0.0350} = 1.036$. Ten extra mg/dL
multiply the odds by $e^{10 \times 0.0350} = 1.42$, an increase of 42%. This
number, $e^{\beta_1}$, is the **odds ratio**.

> **Learning points**
>
> - Logistic regression is linear in the log-odds: $\text{log-odds} = \beta_0 + \beta_1 x$.
> - The slope is the change in log-odds for one unit of the predictor; $e^{\beta_1}$ is the odds ratio.
> - In R the model is about the second level of the outcome factor. Check the levels before you read the signs.

------------------------------------------------------------------------

### Part 3 — From the model to odds and probability

What does the model say about a person with a glucose of 120 mg/dL? **By hand
first**, using the rounded coefficients:

- log-odds: $-5.064 + 0.0350 \times 120 = -0.864$
- odds: $e^{-0.864} = 0.42$
- probability: $\dfrac{0.42}{1 + 0.42} = 0.30$

Now in code, with the coefficients unrounded:

In [ ]:
glucose_value = 120

log_odds_120 = intercept + slope * glucose_value   # the straight line
odds_120     = np.exp(log_odds_120)                # odds
prob_120     = 1 / (1 + np.exp(-log_odds_120))     # probability

print("log-odds:   ", round(log_odds_120, 3))
print("odds:       ", round(odds_120, 3))
print("probability:", round(prob_120, 3))

This is the same as asking the model, with `.predict_proba()`:

In [ ]:
logistic_fit_glucose.predict_proba(pd.DataFrame({"glucose": [120]}))

A person with a glucose of 120 has a predicted probability of diabetes of
0.297: odds of 0.42, or roughly two to five. The first column is the
probability of 0 (no diabetes), the second the probability of 1.

**Your turn.** Do the same for a glucose of 160 mg/dL: first by hand, then by
changing the code.

In [ ]:
# Replace every ______ below, then run this cell.
glucose_value = ______

log_odds_160 = intercept + slope * glucose_value
odds_160     = np.exp(log_odds_160)
prob_160     = 1 / (1 + np.exp(-______))

print("log-odds:   ", round(log_odds_160, 3))
print("odds:       ", round(odds_160, 3))
print("probability:", round(prob_160, 3))

**Predictions for the test set.** `.predict_proba()` gives one column per
class, in the order of `classes_`. Join them to the test data:

In [ ]:
probs = logistic_fit_glucose.predict_proba(test[["glucose"]])

test_preds = test.copy()
test_preds["prob_neg"] = probs[:, 0]
test_preds["prob_pos"] = probs[:, 1]

test_preds[["glucose", "diabetes", "prob_pos", "prob_neg"]].head()

**Question!** Which of the two probability columns do we want?

The probability of diabetes, `prob_pos`. The two columns always add up to 1, so
`prob_neg` holds no extra information; but taking the wrong one turns every
statement about risk upside down, and nothing in the code will warn you.

Here is the fitted curve drawn over the observed outcomes, coded 1 for diabetes
and 0 for none:

In [ ]:
ordered = test_preds.sort_values("glucose")   # a line has to be drawn from left to right

fig, ax = plt.subplots()
ax.scatter(test_preds["glucose"], test_preds["diabetes"], alpha=0.4, color="black")
ax.plot(ordered["glucose"], ordered["prob_pos"], color="red", linewidth=2)
ax.set_yticks([0, 0.5, 1])
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Observed outcome (0/1) and predicted probability")
plt.show()

Most of the people with diabetes (the points at 1) sit where the curve is high,
and most of those without (at 0) where it is low, but the two groups overlap a
great deal between about 100 and 160 mg/dL. The curve crosses 0.5 at a glucose
of about 145.

> **Learning points**
>
> - From the fitted line to a probability: compute the log-odds, exponentiate to get the odds, then $p = \text{odds} / (1 + \text{odds})$.
> - `predict(type = "prob")` and `.predict_proba()` do exactly this calculation for you.
> - Make sure you take the probability of the class you care about.

## Stretch

> **Stretch — where does the curve cross 0.5?**
>
> The predicted probability is 0.5 exactly where the log-odds are 0. Use the two
> coefficients to find that glucose value without plotting anything, then check
> it with `predict()` or `.predict_proba()`.

> **Stretch — add BMI and age**
>
> Fit `diabetes ~ glucose + mass + age` on the same training rows. Does the
> coefficient for glucose change? What does it now mean, in words?